# Formatting Envision files

We will be formatting the envision outputs

In [14]:
import polars as pl
from pathlib import Path
from datetime import datetime
import io
import numpy as np

raw_envision_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/EnVision_2026-04-01._.RAW_DATA")

data_headers = ('Plate', 'Barcode', 'Well')
end_col = 6

In [9]:
def parse_envision(path: Path):
    with open(path, 'r') as f:
        lines = [l.strip() for l in f.readlines()]
    
    header_idx = None
    
    for i, line in enumerate(lines):
        line = line.strip()
        cols = [c.strip() for c in line.split(",")]
        if data_headers[0] in cols and data_headers[1] in cols and data_headers[2] in cols:
            header_idx = i
            break
    
    data_lines = [lines[header_idx]]
    
    for line in lines[header_idx + 1:]:
        if not line.strip():
            break
        first_col = line.split(',')[0].strip()
        if not first_col.isdigit():
            break
        data_lines.append(line)
        
    cleaned = "\n".join(data_lines).replace('=""', '')
    
    return pl.read_csv(io.StringIO(cleaned), null_values=['', 'NA'])

def get_measurement_date(path: Path) -> datetime:
    with open(path, 'r') as f:
        lines = f.readlines()
    header = [c.strip() for c in lines[1].split(',')]
    values = [c.strip().replace('=""', '') for c in lines[2].split(',')]
    
    row = dict(zip(header, values))
    
    return datetime.strptime(row['Measurement date'], '%m/%d/%Y %I:%M:%S %p')
    
        

let's compute real elapsed timepoints from the data

In [18]:
files = sorted(raw_envision_path.iterdir())
file_dates = [(f, get_measurement_date(f)) for f in files]
file_dates.sort(key=lambda x: x[1])

t0 = file_dates[0][1]
timepoints = [(f, round((dt - t0).total_seconds() / 3600 * 2) / 2) for f, dt in file_dates]
print(timepoints)

[(PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/EnVision_2026-04-01._.RAW_DATA/2907.csv'), 0.0), (PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/EnVision_2026-04-01._.RAW_DATA/2908.csv'), 0.5), (PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/EnVision_2026-04-01._.RAW_DATA/2909.csv'), 1.0), (PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/EnVision_2026-04-01._.RAW_DATA/2910.csv'), 1.5), (PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/EnVision_2026-04-01._.RAW_DATA/2911.csv'), 2.0), (PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/EnVision_2026-04-01._.RAW_DATA/2912.csv'), 2.5), (PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/EnVision_2026-04-01._.RAW_DATA/2913.csv'), 3.0), (PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/EnVision_2026-04-01._.RA

In [ ]:
# we will be using actual timepoints in this experiment (not using the sequential ints)

dfs = []

for f, time in timepoints:
    df = (
        parse_envision(f)
        .drop(
            ["MeasTime", 
             "CalcResultI_duplicated_0",
             "Signal_duplicated_0",
             "MeasTime_duplicated_0",
             "_duplicated_0",
             "_duplicated_1",
             "",
             "Barcode", 
             "Flashes/Time",
             "Plate"])
        .with_columns(pl.lit(time).alias("Timepoint"))
    )
    dfs.append(df)
    
final_formatted_df = pl.concat(dfs)

display(final_formatted_df)
final_formatted_df.write_csv("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/enVision_formatted/2026-04-01._.FORMATTED.tsv", separator="\t")

Plate,Well,CalcResultI,Signal,Timepoint
i64,str,i64,i64,f64
1,"""A01""",5000,1000,0.0
1,"""A02""",5100,1020,0.0
1,"""A03""",5400,1080,0.0
1,"""A04""",4800,960,0.0
1,"""A05""",4620,924,0.0
…,…,…,…,…
1,"""P20""",5000,1000,120.0
1,"""P21""",5300,1060,120.0
1,"""P22""",5080,1016,120.0


Let's find the missing time points

In [16]:
expected_interval = 0.5
gaps = []

for i in range(1, len(timepoints)):
    prev_t = timepoints[i - 1][1]
    curr_t = timepoints[i][1]
    diff =  curr_t - prev_t
    if diff > expected_interval:
        missing = list(np.arange(prev_t + expected_interval, curr_t, expected_interval))
        gaps.append({
            'after_file'      : timepoints[i-1][0].stem,
            'before_file'     : timepoints[i][0].stem,
            'after_hour'      : prev_t,
            'before_hour'     : curr_t,
            'missing_hours'   : missing,
            'n_missing'       : len(missing),
        })

# report
if gaps:
    print(f"Found {len(gaps)} gap(s):")
    for g in gaps:
        print(f"  between file {g['after_file']} ({g['after_hour']}h) "
              f"and {g['before_file']} ({g['before_hour']}h) "
              f"— missing timepoints: {g['missing_hours']}")
else:
    print("No gaps found")

Found 2 gap(s):
  between file 3072 (82.5h) and 3073 (83.5h) — missing timepoints: [np.float64(83.0)]
  between file 3098 (96.0h) and 3099 (99.0h) — missing timepoints: [np.float64(96.5), np.float64(97.0), np.float64(97.5), np.float64(98.0), np.float64(98.5)]
